# HSPRN — Colab train (global)

Notebook dùng chung cho mọi thí nghiệm. **Việc của bạn chỉ cần đổi `CONFIG`** (và path Drive nếu khác mặc định).

## Cấu trúc Google Drive (chuẩn)

```text
MyDrive/HSPRN_colab/
├── code.zip                 # zip repo (khuyến nghị)  HOẶC folder HSPRN/
├── prepared_datasets/       # data prepared (CUB/, AWA2/, SUN/, APY/, ...)
│   └── CUB/
│       ├── manifest.yaml
│       ├── metadata/
│       └── raw/...
├── pre-trained/             # CLIP weights (tự tải nếu thiếu)
│   └── ViT-B-32.pt
├── checkpoints/             # output train (tự ghi)
└── logs/
```

Runtime: **GPU**.

## 1. Chỉ cần chỉnh block này

In [ ]:
# ========================= USER CONFIG =========================
# Đổi file YAML trong repo là đủ để đổi thí nghiệm.
CONFIG = "config/cub_image_resnet101_unify_after_llm.yaml"

# Root trên Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/HSPRN_colab"

# Nguồn code: "drive_zip" | "drive_dir" | "git"
CODE_SOURCE = "drive_zip"
GIT_URL = "https://github.com/TongDuyDat/HSPRM.git"
GIT_BRANCH = "labs/after-llm"

# Optional overrides (None = giữ nguyên trong YAML)
EPOCHS = None          # e.g. 80
BATCH_SIZE = None      # e.g. 16 nếu OOM
NUM_WORKERS = 2        # Colab nên 2
SUBSET_SIZE = None     # e.g. 64 smoke
EVAL_SUBSET_SIZE = None
DEVICE = "cuda"        # "cuda" | "cpu"

# True = chỉ smoke 1 epoch + subset nhỏ, không full train
SMOKE_ONLY = False

# Ghi checkpoint/log lên Drive (tránh mất khi disconnect)
SAVE_ON_DRIVE = True
# ===============================================================

## 2. Setup (Drive + code + deps + data) — chạy 1 lần

In [ ]:
import os
import sys
import shutil
import subprocess
import tarfile
import urllib.request
from pathlib import Path

import torch
from google.colab import drive

WORK_DIR = Path("/content/HSPRN")
DRIVE = Path(DRIVE_ROOT)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no-gpu")

drive.mount("/content/drive")
assert DRIVE.exists(), f"Thiếu Drive root: {DRIVE}. Tạo folder rồi chạy lại."

# ---------- code ----------
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

if CODE_SOURCE == "git":
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, str(WORK_DIR)])
elif CODE_SOURCE == "drive_zip":
    zpath = DRIVE / "code.zip"
    assert zpath.is_file(), f"Thiếu {zpath}"
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["unzip", "-q", str(zpath), "-d", str(WORK_DIR)])
    children = [p for p in WORK_DIR.iterdir() if p.name != "__MACOSX"]
    if len(children) == 1 and children[0].is_dir() and not (WORK_DIR / "train.py").exists():
        inner = children[0]
        for item in inner.iterdir():
            shutil.move(str(item), str(WORK_DIR / item.name))
        try:
            inner.rmdir()
        except OSError:
            pass
elif CODE_SOURCE == "drive_dir":
    src = DRIVE / "HSPRN"
    assert src.is_dir(), f"Thiếu {src}"
    shutil.copytree(src, WORK_DIR)
else:
    raise ValueError(CODE_SOURCE)

assert (WORK_DIR / "train.py").is_file(), f"Không thấy train.py trong {WORK_DIR}"
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
print("CWD:", os.getcwd())

# ---------- deps ----------
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "omegaconf", "ftfy", "regex", "timm", "transformers>=4.40,<5",
    "safetensors", "tqdm", "PyYAML", "scipy", "Pillow", "setuptools",
])
from omegaconf import OmegaConf  # noqa: E402
from tqdm.auto import tqdm  # noqa: E402
print("deps OK")

# ---------- link data from Drive ----------
def link_or_copy(name: str, required: bool = False):
    """Symlink Drive/<name> -> WORK_DIR/<name>."""
    dst = WORK_DIR / name
    src = DRIVE / name
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink():
            dst.unlink()
        elif dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.exists():
        dst.symlink_to(src.resolve(), target_is_directory=src.is_dir())
        print(f"link {name} → {src}")
        return True
    if required:
        raise FileNotFoundError(f"Thiếu bắt buộc trên Drive: {src}")
    print(f"skip {name} (không có trên Drive)")
    return False

# prepared_datasets có thể là folder sẵn, hoặc archive
prep_drive = DRIVE / "prepared_datasets"
prep_local = WORK_DIR / "prepared_datasets"
if prep_drive.is_dir():
    link_or_copy("prepared_datasets")
else:
    # thử archive
    archives = [
        DRIVE / "prepared_datasets.tar.gz",
        DRIVE / "prepared_datasets.tgz",
        DRIVE / "prepared_datasets.zip",
        DRIVE / "cub_prepared.tar.gz",
    ]
    found = next((a for a in archives if a.is_file()), None)
    if found is None:
        raise FileNotFoundError(
            f"Thiếu data prepared trên Drive.\n"
            f"Cần folder: {prep_drive}\n"
            f"hoặc archive: prepared_datasets.tar.gz / cub_prepared.tar.gz"
        )
    print("Extracting", found, "…")
    prep_local.mkdir(parents=True, exist_ok=True)
    if found.suffix == ".zip":
        subprocess.check_call(["unzip", "-q", str(found), "-d", str(WORK_DIR)])
    else:
        with tarfile.open(found, "r:*") as tf:
            tf.extractall(WORK_DIR)
    # normalize: ensure WORK_DIR/prepared_datasets exists
    if not prep_local.exists():
        # archive may contain only CUB/
        cand = list(WORK_DIR.rglob("manifest.yaml"))
        if cand:
            pkg = cand[0].parent
            prep_local.mkdir(parents=True, exist_ok=True)
            target = prep_local / pkg.name
            if not target.exists():
                shutil.move(str(pkg), str(target))
    print("prepared_datasets ready:", prep_local)

link_or_copy("pre-trained")
link_or_copy("data")  # optional legacy
link_or_copy("datasets")  # optional

# output dirs on Drive
(DRIVE / "checkpoints").mkdir(parents=True, exist_ok=True)
(DRIVE / "logs").mkdir(parents=True, exist_ok=True)
if SAVE_ON_DRIVE:
    link_or_copy("checkpoints")
    link_or_copy("logs")

print("Setup done.")


## 3. Load config + chuẩn bị weights

Cell này đọc YAML bạn chọn, apply override nhẹ, đảm bảo CLIP weight nếu config cần.

In [ ]:
from tools.utils import load_config

config_path = WORK_DIR / CONFIG
assert config_path.is_file(), f"Không thấy config: {config_path}\nCó sẵn: {[p.name for p in (WORK_DIR/'config').glob('*.yaml')]}"

default_path = WORK_DIR / "config" / "default.yaml"
cfg = load_config(str(config_path), default_path=str(default_path) if default_path.exists() else None)

# Force prepared root local (symlink → Drive)
cfg.data.source = "prepared"
cfg.data.prepared_root = str(WORK_DIR / "prepared_datasets")
cfg.data.num_workers = int(NUM_WORKERS)

if BATCH_SIZE is not None:
    cfg.data.batch_size = int(BATCH_SIZE)
if EPOCHS is not None:
    cfg.training.epochs = int(EPOCHS)
if SUBSET_SIZE is not None:
    cfg.data.subset_size = int(SUBSET_SIZE)
if EVAL_SUBSET_SIZE is not None:
    cfg.data.eval_subset_size = int(EVAL_SUBSET_SIZE)

# Checkpoint/log trên Drive qua symlink ./checkpoints ./logs
if SAVE_ON_DRIVE:
    # giữ tên run theo stem config để không đè lẫn thí nghiệm
    run_name = Path(CONFIG).stem
    cfg.training.save_dir = str(WORK_DIR / "checkpoints" / run_name)
    cfg.training.log_dir = str(WORK_DIR / "logs" / run_name)
    Path(cfg.training.save_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.training.log_dir).mkdir(parents=True, exist_ok=True)

# Ghi runtime config (để train.py / eval.py dùng)
runtime_cfg_path = WORK_DIR / "config" / "_colab_runtime.yaml"
OmegaConf.save(cfg, runtime_cfg_path)
print("Runtime config →", runtime_cfg_path)
print("dataset:", cfg.data.dataset)
print("vision:", cfg.model.vision.backbone_type, "| freeze:", cfg.model.vision.freeze)
print("semantic:", cfg.model.semantic.backbone_type)
print("unify_after_llm:", bool(getattr(getattr(cfg, "experiment", None), "unify_after_llm", False)))
print("batch:", cfg.data.batch_size, "| epochs:", cfg.training.epochs)
print("save_dir:", cfg.training.save_dir)

# prepared package check
ds_name = str(cfg.data.dataset).upper()
pkg = Path(cfg.data.prepared_root) / ds_name
man = pkg / "manifest.yaml"
assert man.is_file(), f"Thiếu package prepared: {pkg} (upload prepared_datasets/{ds_name} lên Drive)"
print("package:", pkg)
print(man.read_text(encoding="utf-8").splitlines()[:12])

# CLIP / pretrained weights helper
CLIP_URLS = {
    "ViT-B/32": (
        "https://openaipublic.azureedge.net/clip/models/"
        "40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt",
        "ViT-B-32.pt",
    ),
    "ViT-B/16": (
        "https://openaipublic.azureedge.net/clip/models/"
        "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f/ViT-B-16.pt",
        "ViT-B-16.pt",
    ),
    "ViT-L/14": (
        "https://openaipublic.azureedge.net/clip/models/"
        "b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt",
        "ViT-L-14.pt",
    ),
}

def ensure_weight(path_or_name: str):
    if not path_or_name:
        return
    p = Path(path_or_name)
    # OpenAI name like ViT-B/32
    if path_or_name in CLIP_URLS:
        url, fname = CLIP_URLS[path_or_name]
        local = WORK_DIR / "pre-trained" / fname
        drive_pt = DRIVE / "pre-trained" / fname
        local.parent.mkdir(parents=True, exist_ok=True)
        if local.is_file():
            print("weight OK:", local)
            return
        if drive_pt.is_file():
            if not (WORK_DIR / "pre-trained").exists():
                (WORK_DIR / "pre-trained").mkdir(parents=True, exist_ok=True)
            if not local.exists():
                # if pre-trained is symlink to drive, file already visible
                if not drive_pt.resolve() == local.resolve():
                    try:
                        local.symlink_to(drive_pt)
                    except FileExistsError:
                        pass
            print("weight from Drive:", drive_pt)
            return
        print("Downloading", fname, "…")
        class T(tqdm):
            def update_to(self, b=1, bsize=1, tsize=None):
                if tsize is not None:
                    self.total = tsize
                self.update(b * bsize - self.n)
        with T(unit="B", unit_scale=True, miniters=1, desc=fname) as t:
            urllib.request.urlretrieve(url, local, reporthook=t.update_to)
        drive_pt.parent.mkdir(parents=True, exist_ok=True)
        if not drive_pt.exists():
            shutil.copy2(local, drive_pt)
        print("cached →", drive_pt)
        return

    # local path relative to repo
    if not p.is_absolute():
        p = WORK_DIR / p
    if p.is_file():
        print("weight OK:", p)
        return
    # try Drive pre-trained by filename
    drive_pt = DRIVE / "pre-trained" / p.name
    if drive_pt.is_file():
        p.parent.mkdir(parents=True, exist_ok=True)
        if not p.exists():
            try:
                p.symlink_to(drive_pt.resolve())
            except OSError:
                shutil.copy2(drive_pt, p)
        print("weight from Drive:", drive_pt)
        return
    # map known filenames
    for key, (url, fname) in CLIP_URLS.items():
        if p.name == fname or path_or_name.endswith(fname):
            ensure_weight(key)
            # point expected path
            src = WORK_DIR / "pre-trained" / fname
            if src.is_file() and not p.exists():
                p.parent.mkdir(parents=True, exist_ok=True)
                try:
                    p.symlink_to(src.resolve())
                except OSError:
                    shutil.copy2(src, p)
            return
    print("WARNING: weight not found:", path_or_name)

# semantic clip
sem = getattr(cfg.model, "semantic", None)
if sem is not None:
    ensure_weight(str(getattr(sem, "clip_model", "") or ""))
    ensure_weight(str(getattr(sem, "pretrained_weights", "") or ""))

# vision clip name if any
vis = getattr(cfg.model, "vision", None)
if vis is not None:
    ensure_weight(str(getattr(vis, "clip_model", "") or ""))

print("Weights ready.")

## 4. Train

Chạy cell này sau khi đã chọn `CONFIG`. Muốn đổi thí nghiệm: sửa `CONFIG` ở cell 1 → chạy lại cell 3 → cell 4.

In [ ]:
runtime_cfg_path = WORK_DIR / "config" / "_colab_runtime.yaml"
assert runtime_cfg_path.is_file(), "Chạy cell Load config trước."

device = DEVICE if (DEVICE != "cuda" or torch.cuda.is_available()) else "cpu"
if device == "cpu":
    print("WARNING: training on CPU")

cmd = [
    sys.executable, "train.py",
    "--config", str(runtime_cfg_path),
    "--prepared_root", str(WORK_DIR / "prepared_datasets"),
    "--device", device,
    "--num_workers", str(NUM_WORKERS),
]

if SMOKE_ONLY:
    cmd += ["--epochs", "1", "--subset_size", "64", "--eval_subset_size", "32"]
    print("SMOKE_ONLY=True")
else:
    if EPOCHS is not None:
        cmd += ["--epochs", str(int(EPOCHS))]
    if SUBSET_SIZE is not None:
        cmd += ["--subset_size", str(int(SUBSET_SIZE))]
    if EVAL_SUBSET_SIZE is not None:
        cmd += ["--eval_subset_size", str(int(EVAL_SUBSET_SIZE))]

print(" ".join(cmd))
subprocess.check_call(cmd)
print("Train done. Checkpoints:", cfg.training.save_dir)

## 5. Eval (optional)

In [ ]:
save_dir = Path(cfg.training.save_dir)
ckpts = sorted(save_dir.rglob("best.pth"))
if not ckpts:
    # fallback: any best under Drive checkpoints
    ckpts = sorted((DRIVE / "checkpoints").rglob("best.pth"))

print("checkpoints:")
for p in ckpts:
    print(" -", p)
assert ckpts, f"Không thấy best.pth trong {save_dir}"
ckpt = ckpts[-1]
print("Using", ckpt)

device = DEVICE if (DEVICE != "cuda" or torch.cuda.is_available()) else "cpu"
cmd = [
    sys.executable, "eval.py",
    "--config", str(runtime_cfg_path),
    "--ckpt", str(ckpt),
    "--prepared_root", str(WORK_DIR / "prepared_datasets"),
    "--device", device,
    "--num_workers", str(NUM_WORKERS),
]
print(" ".join(cmd))
subprocess.check_call(cmd)

## Đổi thí nghiệm nhanh

Chỉ sửa `CONFIG` rồi chạy lại **cell 3 → 4** (không cần setup lại):

```python
CONFIG = "config/cub_image_resnet101_unify_after_llm.yaml"   # Plan3 CUB ResNet101
CONFIG = "config/cub_image_resnet50_train.yaml"              # CUB ResNet50
CONFIG = "config/cub_image_clip_vit_b16_unify_after_llm.yaml"
CONFIG = "config/awa2_image_resnet101_train.yaml"
CONFIG = "config/sun_image_resnet50_train.yaml"
```

Dataset trong YAML phải có package tương ứng:
`Drive/HSPRN_colab/prepared_datasets/<DATASET>/`.

### Pack code từ máy local

```bash
cd /path/to/HSPRN
zip -r code.zip . \
  -x 'prepared_datasets/*' -x 'datasets/*' -x 'data/*' \
  -x 'checkpoints/*' -x 'logs/*' -x 'pre-trained/*' \
  -x '.git/*' -x '**/__pycache__/*' -x '*.pt' -x '*.pth'
```

### Pack data

```bash
tar -cf prepared_datasets.tar.gz prepared_datasets
# hoặc từng bộ: tar -cf cub_prepared.tar.gz prepared_datasets/CUB
```